In [1]:
import csv
import glob
import os
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
os.chdir(r"../Data/Twitter/cleaned")
filenames = glob.glob("*cleaned_sentiment*.csv")

combined_df = pd.DataFrame()

for file in filenames:
    df = pd.read_csv(file)
    df['Source'] = ['Twitter'] * df.shape[0]
    df['Stock'] = [file.split('_')[0]] * df.shape[0] 
    combined_df = pd.concat([combined_df, df])

# changing flaire sentiment score to match flaire_sentiment
mask = (combined_df['flair_sentiment'] == 'NEGATIVE')
combined_df['flair_sentiment_score'][mask] *= -1

# removed number of comments and retweets as well
combined_df.drop(columns = ['Unnamed: 0', 'Unnamed: 0.1', 'Username', 'Tweet', 'cleaned Tweet', 'flair_sentiment', 'vader_sentiment', 'Number of Comments', 'Number of Retweets'], inplace = True)

combined_df.head()

,Datetime,Number of Followers,Number of Comments,Number of Retweets,Number of Likes,flair_sentiment_score,vader_score,finbert_sentiment,finbert_score,Source,Stock
0,2021-09-30 18:01:19+00:00,559219,215,442,2532,-0.9999,-0.0772,neutral,0.820654,Twitter,AAPL
1,2021-09-30 15:30:03+00:00,213150,6,17,215,0.7746,0.5267,neutral,0.940475,Twitter,AAPL
2,2021-09-30 13:36:10+00:00,33383,21,68,361,-0.9786,0.0000,neutral,0.831750,Twitter,AAPL
3,2021-09-30 10:19:31+00:00,14836,22,36,237,0.9617,0.3182,neutral,0.935794,Twitter,AAPL
4,2021-09-29 23:24:02+00:00,128553,22,36,224,-0.6766,0.4019,neutral,0.931196,Twitter,AAPL


# Normalizing Data based on day

In [21]:
import datetime

In [7]:
combined_df['Datetime'] = pd.to_datetime(combined_df['Datetime']).dt.date

combined_df.head()

,Datetime,Number of Followers,Number of Comments,Number of Retweets,Number of Likes,flair_sentiment_score,vader_score,finbert_sentiment,finbert_score,Source,Stock
0,2021-09-30,559219,215,442,2532,-0.9999,-0.0772,neutral,0.820654,Twitter,AAPL
1,2021-09-30,213150,6,17,215,0.7746,0.5267,neutral,0.940475,Twitter,AAPL
2,2021-09-30,33383,21,68,361,-0.9786,0.0000,neutral,0.831750,Twitter,AAPL
3,2021-09-30,14836,22,36,237,0.9617,0.3182,neutral,0.935794,Twitter,AAPL
4,2021-09-29,128553,22,36,224,-0.6766,0.4019,neutral,0.931196,Twitter,AAPL


In [9]:

dates = combined_df['Datetime'].unique()
dates.sort()

followers = combined_df.groupby(['Datetime', 'Stock'])['Number of Followers'].max()
likes = combined_df.groupby(['Datetime', 'Stock'])['Number of Likes'].max()
variable_counts = list(zip(followers, likes))


daily_max = {dates[i]: variable_counts[i] for i in range(len(dates))}    

In [17]:
#creating dictionary column name : [datetime.date, stock] : max value
#dictionary within a dictionary, first key = column name, second key = [datetime.date, stock]

combined_df.groupby(['Datetime', 'Stock']).max()

daily_max = combined_df.groupby(['Datetime', 'Stock']).max().apply(lambda x : x .to_list()).to_dict()

{'Number of Followers': {(datetime.date(2021, 1, 1), 'AAPL'): 68944,
  (datetime.date(2021, 1, 1), 'AMZN'): 33317,
  (datetime.date(2021, 1, 1), 'ARKK'): 33151,
  (datetime.date(2021, 1, 1), 'GOOGL'): 33308,
  (datetime.date(2021, 1, 1), 'MSFT'): 33315,
  (datetime.date(2021, 1, 1), 'TSLA'): 589808,
  (datetime.date(2021, 1, 1), 'VXRT'): 44380,
  (datetime.date(2021, 1, 2), 'AAPL'): 404183,
  (datetime.date(2021, 1, 2), 'AMZN'): 405443,
  (datetime.date(2021, 1, 2), 'ARKK'): 400,
  (datetime.date(2021, 1, 2), 'GOOGL'): 405384,
  (datetime.date(2021, 1, 2), 'MSFT'): 405422,
  (datetime.date(2021, 1, 2), 'Russell 1000'): 309740,
  (datetime.date(2021, 1, 2), 'TSLA'): 600061,
  (datetime.date(2021, 1, 2), 'VXRT'): 44380,
  (datetime.date(2021, 1, 3), 'AAPL'): 601323,
  (datetime.date(2021, 1, 3), 'AMZN'): 33317,
  (datetime.date(2021, 1, 3), 'ARKK'): 21215,
  (datetime.date(2021, 1, 3), 'MSFT'): 212991,
  (datetime.date(2021, 1, 3), 'TSLA'): 243474,
  (datetime.date(2021, 1, 4), 'AAPL'): 

<b>sample input to get max followers per day:</b> daily_max['Number of Followers'][(datetime.date(2021, 1, 1), 'AAPL')]

<b>sample output:</b> 68944

# Applying Weighted Sentiment for weekends

In [ ]:
os.chdir('../../../Sentiment Calculator')
%run Weighted_Sentiment.ipynb

In [ ]:
data = group_by_date(combined_df)
weighted_sent = pd.DataFrame(weighted_weekends(data), columns = ['date', 'flair_weighted', 'vader_weighted', 'finbert_weighted'])
weighted_sent